# Feature Swap Comparison (2.8 s)

Subject CC = mean over 5 fingers. Grand Average (GA) = mean over subjects. Relative change = `(swap - original) / original`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("results/o5")
SEED, WIN = 42, 2.8
SUBJECTS = {
    "BCIIV": ("sub1", "sub2", "sub3"),
    "Stanford": ("bp", "cc", "ht", "jc", "jp", "mv", "wc", "wm", "zt"),
}
MODELS = ("HiLoFuseNet", "LSTM", "MLP")

def load_result(dataset, model, swapped=False):
    tag = "_featureSwap" if swapped else ""
    folder = "featureSwap" if swapped else "varyingWindow"
    path = ROOT / folder / f"{dataset}_{model}{tag}_mse_seed{SEED}_win{WIN}_o5_cc.npy"
    values = np.load(path)
    assert values.shape == (5, len(SUBJECTS[dataset])) and np.isfinite(values).all()
    return values

def build_table(dataset):
    rows = []
    for model in MODELS:
        original, swap = load_result(dataset, model), load_result(dataset, model, True)
        for index, subject in enumerate(SUBJECTS[dataset]):
            old, new = original[:, index].mean(), swap[:, index].mean()
            rows.append([subject, model, old, new, new - old, (new - old) / old, False])
        old, new = original.mean(), swap.mean()
        rows.append(["Grand Average", model, old, new, new - old, (new - old) / old, True])
    return pd.DataFrame(rows, columns=["Subject", "Model", "Original CC", "Feature swap CC", "Delta CC", "Relative change", "is_ga"])

def show(table):
    output = table.drop(columns="is_ga", errors="ignore").copy()
    for column in ["Original CC", "Feature swap CC"]:
        output[column] = output[column].map(lambda value: f"{value:.3f}")
    output["Delta CC"] = output["Delta CC"].map(lambda value: f"{value:+.3f}")
    output["Relative change"] = output["Relative change"].map(lambda value: f"{value:+.1%}")
    return output

tables = {dataset: build_table(dataset) for dataset in SUBJECTS}
ga_rows = []
for model in MODELS:
    all_subjects = pd.concat([
        table.loc[(~table.is_ga) & (table.Model == model)]
        for table in tables.values()
    ], ignore_index=True)
    old, new = all_subjects["Original CC"].mean(), all_subjects["Feature swap CC"].mean()
    ga_rows.append([model, len(all_subjects), old, new, new - old, (new - old) / old])
ga = pd.DataFrame(ga_rows, columns=["Model", "N subjects", "Original CC", "Feature swap CC", "Delta CC", "Relative change"])

## BCIIV

In [2]:
show(tables["BCIIV"])

,Subject,Model,Original CC,Feature swap CC,Delta CC,Relative change
0,sub1,HiLoFuseNet,0.644,0.495,-0.150,-23.2%
1,sub2,HiLoFuseNet,0.654,0.445,-0.210,-32.0%
2,sub3,HiLoFuseNet,0.723,0.657,-0.066,-9.2%
3,Grand Average,HiLoFuseNet,0.674,0.532,-0.142,-21.1%
4,sub1,LSTM,0.544,0.493,-0.051,-9.3%
5,sub2,LSTM,0.542,0.431,-0.112,-20.6%
6,sub3,LSTM,0.648,0.689,+0.041,+6.4%
7,Grand Average,LSTM,0.578,0.538,-0.040,-7.0%
8,sub1,MLP,0.436,0.419,-0.017,-3.9%
9,sub2,MLP,0.428,0.301,-0.127,-29.7%


## Stanford

In [3]:
show(tables["Stanford"])

,Subject,Model,Original CC,Feature swap CC,Delta CC,Relative change
0,bp,HiLoFuseNet,0.633,0.480,-0.152,-24.1%
1,cc,HiLoFuseNet,0.673,0.649,-0.024,-3.5%
2,ht,HiLoFuseNet,0.472,0.255,-0.217,-46.0%
3,jc,HiLoFuseNet,0.652,0.468,-0.184,-28.2%
4,jp,HiLoFuseNet,0.552,0.502,-0.050,-9.0%
5,mv,HiLoFuseNet,0.631,0.536,-0.095,-15.1%
6,wc,HiLoFuseNet,0.479,0.355,-0.124,-25.8%
7,wm,HiLoFuseNet,0.293,0.155,-0.138,-47.0%
8,zt,HiLoFuseNet,0.641,0.510,-0.131,-20.4%
9,Grand Average,HiLoFuseNet,0.558,0.435,-0.124,-22.2%


## Grand Average (all 12 subjects)

BCIIV and Stanford subjects are pooled; every subject receives equal weight.

In [4]:
show(ga)

,Model,N subjects,Original CC,Feature swap CC,Delta CC,Relative change
0,HiLoFuseNet,12,0.587,0.459,-0.128,-21.9%
1,LSTM,12,0.510,0.443,-0.066,-13.0%
2,MLP,12,0.419,0.330,-0.088,-21.1%
